# `XGBoost for Regression Problem`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [2]:
df = pd.read_csv('../09. RandomForest/cardekho_imputated.csv',index_col=[0])

In [3]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


### **Feature Engineering**
#### `Data Cleaning`
#### Handling Missing Values
* Handling Missing Values
* Handling Duplicates
* Check data type 
* Understand the dataset


In [4]:
## Checking missing values 
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

1. `The dataset has no Missing values.`

In [5]:
### Remove unnecessary Columns
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace=True)

In [6]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [7]:
### to check the unique models
df['model'].unique()

array(['Alto', 'Grand', 'i20', 'Ecosport', 'Wagon R', 'i10', 'Venue',
       'Swift', 'Verna', 'Duster', 'Cooper', 'Ciaz', 'C-Class', 'Innova',
       'Baleno', 'Swift Dzire', 'Vento', 'Creta', 'City', 'Bolero',
       'Fortuner', 'KWID', 'Amaze', 'Santro', 'XUV500', 'KUV100', 'Ignis',
       'RediGO', 'Scorpio', 'Marazzo', 'Aspire', 'Figo', 'Vitara',
       'Tiago', 'Polo', 'Seltos', 'Celerio', 'GO', '5', 'CR-V',
       'Endeavour', 'KUV', 'Jazz', '3', 'A4', 'Tigor', 'Ertiga', 'Safari',
       'Thar', 'Hexa', 'Rover', 'Eeco', 'A6', 'E-Class', 'Q7', 'Z4', '6',
       'XF', 'X5', 'Hector', 'Civic', 'D-Max', 'Cayenne', 'X1', 'Rapid',
       'Freestyle', 'Superb', 'Nexon', 'XUV300', 'Dzire VXI', 'S90',
       'WR-V', 'XL6', 'Triber', 'ES', 'Wrangler', 'Camry', 'Elantra',
       'Yaris', 'GL-Class', '7', 'S-Presso', 'Dzire LXI', 'Aura', 'XC',
       'Ghibli', 'Continental', 'CR', 'Kicks', 'S-Class', 'Tucson',
       'Harrier', 'X3', 'Octavia', 'Compass', 'CLS', 'redi-GO', 'Glanza',
       

In [8]:
## Getting different types of Features 

num_features = [feature for feature in df.columns if df[feature].dtype !='O']
print('Total Numerical Feature: ',len(num_features))

cat_features = [feature for feature in df.columns if df[feature].dtype =='O']
print('Total Categorical Feature: ',len(cat_features))

## For Descrete and Continuous Feature 
discrete_feature = [feature for feature in num_features if len(df[feature].unique()) <=25]
print('Total Discrete Feature: ',len(discrete_feature))

continuous_feature = [feature for feature in num_features if feature not in discrete_feature]
print('Total Continuous Feature: ',len(continuous_feature))


Total Numerical Feature:  7
Total Categorical Feature:  4
Total Discrete Feature:  2
Total Continuous Feature:  5


### **Spliting Data**

In [9]:
## Dividing data into independent and dependent features
X = df.drop(['selling_price'],axis=1)
y = df['selling_price']

In [10]:
### Indpendent and dependent features 
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'],axis=1)
y = df['selling_price']

In [11]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [12]:
y.head()

0    120000
1    550000
2    215000
3    226000
4    570000
Name: selling_price, dtype: int64

### `Feature Encoding and Scaling`

**One Hot Encoding for columns which had lesser unique values and not ordinal**
* One hot encoding is a process by which categorical variables are converted into a form this could be provided to ML algorithms to do a better job in prediction

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [14]:
X

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5
...,...,...,...,...,...,...,...,...,...,...
19537,117,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5
19540,42,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7
19541,77,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5
19542,114,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7


In [15]:
## now we check the categories in the remaining categorical columns 
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

* This is show that these each column has not too many category so, we can use `OneHotEncoder`

In [16]:
### Create a column transformer with 3 types of transformers 
num_features = X.select_dtypes(exclude='O').columns
onehot_columns = ['seller_type','fuel_type','transmission_type']

from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder",oh_transformer,onehot_columns),
        ("StandardScaler",numeric_transformer,num_features)
    ],remainder='passthrough'
)


In [17]:
X = preprocessor.fit_transform(X)

In [18]:
X

array([[ 1.        ,  0.        ,  0.        , ..., -1.32425883,
        -1.26335238, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43257082, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.47911321, -0.40302241],
       ...,
       [ 0.        ,  0.        ,  1.        , ...,  0.02291783,
         0.06822523, -0.40302241],
       [ 0.        ,  0.        ,  1.        , ...,  1.32979434,
         0.91715831,  2.07344426],
       [ 0.        ,  0.        ,  0.        , ...,  0.02099878,
         0.39588361, -0.40302241]], shape=(15411, 14))

In [19]:
## if i like to see them in dataframe form 
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


In [20]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=.2,random_state=42)

In [21]:
X_train

array([[ 0.        ,  0.        ,  1.        , ...,  1.75390551,
         2.66249771, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55087963,
        -0.38602844, -0.40302241],
       [ 0.        ,  0.        ,  1.        , ...,  0.89033072,
         3.27453006, -0.40302241],
       ...,
       [ 1.        ,  0.        ,  0.        , ..., -0.9366097 ,
        -0.78070786, -0.40302241],
       [ 0.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43582879, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.04616815,
         0.06194201, -0.40302241]], shape=(12328, 14))

# `Model Training and Model Selection`

In [22]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso,Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
## XGBoost Regressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

##### `Evaluation Function`

In [23]:
### Creating a Function to Evaluate Model
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true,predicted)
    mse = mean_squared_error(true,predicted)
    rmse = np.sqrt(mean_squared_error(true,predicted))
    score = r2_score(true,predicted)
    return mae, rmse, score

In [24]:
## Model Training 
models = {
    'Linear Regression': LinearRegression(),
    'Lasso':Lasso(),
    'Ridge':Ridge(),
    'K-Neighbors Regressor':KNeighborsRegressor(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'Gradient Bossting':GradientBoostingRegressor(),
    'XGBRegressor': XGBRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)     ### Model Training
    
    ## Making prediction for Training and Test Data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    ## Evaluate Train and Test Data 
    model_train_mae, model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)
    
    model_test_mae, model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model Performance for Training set')
    print('- Root Mean Squared Error: {:.4f}'.format(model_train_rmse))
    print('- Mean Absolute Error: {:.4f}'.format(model_train_mae))
    print('- R2-Score: {:.4f}'.format(model_train_r2))
    
    print('------------------------------------')
    
    print('Model Performance for Test set')
    print('- Root Mean Squared Error: {:.4f}'.format(model_test_rmse))
    print('- Mean Absolute Error: {:.4f}'.format(model_test_mae))
    print('- R2-Score: {:.4f}'.format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model Performance for Training set
- Root Mean Squared Error: 553855.6665
- Mean Absolute Error: 268101.6071
- R2-Score: 0.6218
------------------------------------
Model Performance for Test set
- Root Mean Squared Error: 502543.5930
- Mean Absolute Error: 279618.5794
- R2-Score: 0.6645


Lasso
Model Performance for Training set
- Root Mean Squared Error: 553855.6710
- Mean Absolute Error: 268099.2226
- R2-Score: 0.6218
------------------------------------
Model Performance for Test set
- Root Mean Squared Error: 502542.6696
- Mean Absolute Error: 279614.7461
- R2-Score: 0.6645


Ridge
Model Performance for Training set
- Root Mean Squared Error: 553856.3160
- Mean Absolute Error: 268059.8015
- R2-Score: 0.6218
------------------------------------
Model Performance for Test set
- Root Mean Squared Error: 502533.8230
- Mean Absolute Error: 279557.2169
- R2-Score: 0.6645


K-Neighbors Regressor
Model Performance for Training set
- Root Mean Squared Error: 325873.0267
-

## Insights
`As three Algorithms are performing well, we need to do Hyperparameter tuning all of them `
1. `Random Forest --->92.81`
2. `KNN ---->91.50`
3. `Gradient Boosting---> 0.9109`
4. `XGB Regressor---> 91.91`

In [29]:
### Initializing few parameter for Hyperparameter tuning 

rf_params = {
            "max_depth":[5,8,15,None,10],
            "max_features":[5,7,'auto',8],
            "min_samples_split":[2,8,15,20],
            "n_estimators":[100,200,500,1000]
}
xgb_params = {
    'learning_rate':[0.1,0.4,0.3,1,2],
    'n_estimators':[100,500,200,40,50],
    'max_depth':[5,8,12,20,30],
    'colsample_bytree':[0.5,0.8,1,0.3,0.4]
}

In [30]:
## Models list for Hyperparameter Tuning
randomcv_models =[
    # ('RF',RandomForestRegressor(),rf_params),
    ('XGB',XGBRegressor(),xgb_params)
]

In [31]:
randomcv_models

[('XGB',
  XGBRegressor(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...),
  {'learning_rate': [0.1, 0.4, 0.3, 1, 2],
   'n_estimators': [100, 500, 200, 40, 50],
   'max_depth': [5, 8, 12, 20, 30],
   'colsample_bytree': [0.5, 0.8, 1, 0.3, 0.4]})]

#### `Hyperparameter Tuning`

In [33]:
from sklearn.model_selection import RandomizedSearchCV

model_param ={}
for name,model,params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                param_distributions=params,
                                n_iter=100,
                                cv=3,
                                verbose=2)
    
    random.fit(X_train,y_train)
    model_param[name]=random.best_params_
    
for model_name in model_param:
    print(f"--------------Best Params for {model_name} ----------------")
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END colsample_bytree=0.4, learning_rate=0.4, max_depth=8, n_estimators=50; total time=   0.0s
[CV] END colsample_bytree=0.4, learning_rate=0.4, max_depth=8, n_estimators=50; total time=   0.0s
[CV] END colsample_bytree=0.4, learning_rate=0.4, max_depth=8, n_estimators=50; total time=   0.0s
[CV] END colsample_bytree=0.8, learning_rate=2, max_depth=20, n_estimators=500; total time=   2.4s
[CV] END colsample_bytree=0.8, learning_rate=2, max_depth=20, n_estimators=500; total time=   2.9s
[CV] END colsample_bytree=0.8, learning_rate=2, max_depth=20, n_estimators=500; total time=   3.5s
[CV] END colsample_bytree=0.5, learning_rate=1, max_depth=30, n_estimators=50; total time=   4.2s
[CV] END colsample_bytree=0.5, learning_rate=1, max_depth=30, n_estimators=50; total time=   4.5s
[CV] END colsample_bytree=0.5, learning_rate=1, max_depth=30, n_estimators=50; total time=   4.5s
[CV] END colsample_bytree=0.3, learning_rate=0.3,

#### `Training Both Models  on Best parameters `

In [35]:
## Model Training 
models = {
    'Random Forest':RandomForestRegressor(n_estimators=200,min_samples_split=2,max_features=8,max_depth=15),
    'XGBoost Regressor':XGBRegressor(n_estimators=400,max_depth=5,learning_rate=.4,colsample_bytree=0.5)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)     ### Model Training
    
    ## Making prediction for Training and Test Data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    ## Evaluate Train and Test Data 
    model_train_mae, model_train_rmse,model_train_r2 = evaluate_model(y_train,y_train_pred)
    
    model_test_mae, model_test_rmse,model_test_r2 = evaluate_model(y_test,y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model Performance for Training set')
    print('- Root Mean Squared Error: {:.4f}'.format(model_train_rmse))
    print('- Mean Absolute Error: {:.4f}'.format(model_train_mae))
    print('- R2-Score: {:.4f}'.format(model_train_r2))
    
    print('------------------------------------')
    
    print('Model Performance for Test set')
    print('- Root Mean Squared Error: {:.4f}'.format(model_test_rmse))
    print('- Mean Absolute Error: {:.4f}'.format(model_test_mae))
    print('- R2-Score: {:.4f}'.format(model_test_r2))
    
    print('='*35)
    print('\n')

Random Forest
Model Performance for Training set
- Root Mean Squared Error: 133835.3528
- Mean Absolute Error: 54090.4633
- R2-Score: 0.9779
------------------------------------
Model Performance for Test set
- Root Mean Squared Error: 212160.8772
- Mean Absolute Error: 97854.3590
- R2-Score: 0.9402


XGBoost Regressor
Model Performance for Training set
- Root Mean Squared Error: 73412.8856
- Mean Absolute Error: 52801.0273
- R2-Score: 0.9934
------------------------------------
Model Performance for Test set
- Root Mean Squared Error: 257450.6419
- Mean Absolute Error: 103099.8672
- R2-Score: 0.9120




### WE have seen the implimentation of XGBosst Regressor

# Now here our Supervised Learning get Completed. 